[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/fingerprint-similarity-guide/blob/main/notebooks/11_speed_and_scale.ipynb)

# 11 · Speed & Scale

*Fingerprint & Similarity Guide · Hamza Hentabli, Ph.D. · ~40 min*

Everything so far used thousands of molecules. Real libraries have millions. At that scale *how* you compute similarity matters as much as the metric. We measure RDKit's bulk popcount routines against a naive Python loop, show the packed-bit trick behind the speed, and point to FPSim2 for million-compound search (kept optional so this notebook always runs).

**Learning objectives.** By the end of this notebook you will be able to:

- use RDKit bulk similarity instead of Python loops and measure the speedup
- understand why Tanimoto reduces to fast integer **popcounts**
- estimate how screening time scales with library size
- know when to reach for FPSim2 / on-disk fingerprint databases (optional)

In [1]:
# Setup: on Google Colab this installs dependencies and clones the repo so the
# cached data and the shared `fsg` helper module are available. Running locally,
# install requirements.txt once and this cell just locates the files.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
OFFLINE = os.environ.get("FSG_OFFLINE", "0") == "1"   # CI sets this; no notebook needs the network
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rdkit", "scikit-learn", "scipy", "pandas", "matplotlib"], check=True)
    if not os.path.isdir("fingerprint-similarity-guide"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/fingerprint-similarity-guide.git"], check=True)
    ROOT = "fingerprint-similarity-guide"
else:
    ROOT = ".." if os.path.isdir("../data") else "."
DATA_DIR = os.path.join(ROOT, "data")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import fsg   # shared fingerprint factory + metrics + evaluation

print("Colab:", IN_COLAB, "| offline:", OFFLINE, "| data:", DATA_DIR)
print("Fingerprints available:", ", ".join(fsg.FP_NAMES))

Colab: False | offline: True | data: ../data
Fingerprints available: ECFP4-1024, ECFP4-2048, ECFP6-2048, ECFP4-counts, FCFP4-2048, RDKit, AtomPair, Torsion, MACCS, Avalon


In [2]:
import numpy as np, pandas as pd, time
from rdkit import DataStructs
dec = fsg.load_mols(pd.read_csv(f"{DATA_DIR}/dude_egfr_decoys.csv"))
lib = fsg.make_fps("ECFP4-2048", list(dec["mol"]))
query = lib[0]
print(f"library size: {len(lib)} fingerprints")

library size: 3000 fingerprints


## 1. Bulk vs naive loop

RDKit's `BulkTanimotoSimilarity` runs the whole query-vs-library comparison in C. Compare it to the same thing in a Python loop.

In [3]:
t0 = time.perf_counter()
naive = [DataStructs.TanimotoSimilarity(query, fp) for fp in lib]
t_naive = time.perf_counter() - t0

t0 = time.perf_counter()
bulk = DataStructs.BulkTanimotoSimilarity(query, lib)
t_bulk = time.perf_counter() - t0

assert np.allclose(naive, bulk)
print(f"naive Python loop : {t_naive*1000:7.1f} ms")
print(f"BulkTanimoto (C)  : {t_bulk*1000:7.1f} ms")
print(f"speedup           : {t_naive/t_bulk:7.1f}x")

naive Python loop :     3.7 ms
BulkTanimoto (C)  :     0.4 ms
speedup           :    10.0x


The bulk routine is many times faster for identical results — always prefer it. The gap widens with library size.

## 2. Why Tanimoto is cheap: popcounts

A binary fingerprint is a packed bit array. Tanimoto = |A AND B| / |A OR B|, and counting set bits (**popcount**) is a single fast CPU instruction. We reproduce it with numpy to demystify what RDKit does internally.

In [4]:
def to_np(fp):
    a = np.zeros((fp.GetNumBits(),), dtype=np.uint8); DataStructs.ConvertToNumpyArray(fp, a); return a
A, B = to_np(lib[0]), to_np(lib[1])
inter = int(np.bitwise_and(A, B).sum())
union = int(np.bitwise_or(A, B).sum())
print("numpy popcount Tanimoto:", round(inter/union, 4))
print("RDKit Tanimoto        :", round(DataStructs.TanimotoSimilarity(lib[0], lib[1]), 4))

numpy popcount Tanimoto: 0.0213
RDKit Tanimoto        : 0.0213


## 3. How screening time scales

Similarity search is linear in library size for a single query. We time a few sizes to see it.

In [5]:
rows = []
for n in [500, 1000, 2000, len(lib)]:
    sub = lib[:n]
    t0 = time.perf_counter()
    for _ in range(5):
        DataStructs.BulkTanimotoSimilarity(query, sub)
    dt = (time.perf_counter() - t0) / 5
    rows.append((n, round(dt*1000, 2), round(n/dt/1e6, 2)))
pd.DataFrame(rows, columns=["library_size","ms_per_query","million_comparisons_per_sec"])

,library_size,ms_per_query,million_comparisons_per_sec
0,500,0.04,12.41
1,1000,0.06,16.08
2,2000,0.13,15.76
3,3000,0.18,16.60


Throughput is roughly constant per comparison, so a single-query screen of 1,000,000 compounds is still well under a second. The challenge at scale is **memory and repeated queries**, which is where dedicated libraries help.

## 4. Optional: FPSim2 for million-compound databases

`FPSim2` stores fingerprints in an on-disk HDF5 file and uses SIMD popcounts for very fast, repeated similarity search over millions of molecules. It is an optional dependency; this cell uses it if present and otherwise explains when you would add it.

In [6]:
try:
    from FPSim2 import FPSim2Engine   # pip install FPSim2
    print("FPSim2 available — for production, build an on-disk DB once and query it repeatedly:")
    print("  from FPSim2.io import create_db_file; create_db_file(smi_list, 'lib.h5', 'Morgan', {'radius':2,'nBits':2048})")
    print("  eng = FPSim2Engine('lib.h5'); eng.similarity(query_smiles, 0.7)")
except Exception as e:
    print("FPSim2 not installed (optional):", type(e).__name__)
    print("Reach for FPSim2 (or chemfp) when you repeatedly search >~1M compounds and RDKit bulk")
    print("in RAM becomes the bottleneck. For the thousands-scale work in this course, RDKit bulk is plenty.")

FPSim2 not installed (optional): ModuleNotFoundError
Reach for FPSim2 (or chemfp) when you repeatedly search >~1M compounds and RDKit bulk
in RAM becomes the bottleneck. For the thousands-scale work in this course, RDKit bulk is plenty.


## Summary

- Always use **bulk** similarity, not Python loops — same answer, large speedup.
- Tanimoto is fast because it is just **popcounts** on packed bits.
- Single-query search is linear and handles ~1M compounds in well under a second.
- For repeated million-compound search, add **FPSim2 / chemfp** (optional).


### ✏️ Exercise 1

Measure the bulk-vs-naive speedup on just the first 500 fingerprints. Is the speedup smaller than on the full library? Why?

<details>
<summary><b>Show solution</b></summary>

```python
sub = lib[:500]
t0=time.perf_counter(); [DataStructs.TanimotoSimilarity(query,fp) for fp in sub]; tn=time.perf_counter()-t0
t0=time.perf_counter(); DataStructs.BulkTanimotoSimilarity(query,sub); tb=time.perf_counter()-t0
print(round(tn/tb,1), "x (fixed call overhead matters more on small inputs)")
```

</details>

### ✏️ Exercise 2

Verify numpy popcount Tanimoto matches RDKit for 5 random library pairs.

<details>
<summary><b>Show solution</b></summary>

```python
import random
for _ in range(5):
    a,b=random.sample(range(len(lib)),2)
    A,B=to_np(lib[a]),to_np(lib[b])
    t=int(np.bitwise_and(A,B).sum())/int(np.bitwise_or(A,B).sum())
    print(round(t,4), round(DataStructs.TanimotoSimilarity(lib[a],lib[b]),4))
```

</details>